In [1]:
import os
import GEOparse
import pandas as pd

# Get GEO_ID from parent folder name
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))

gse = GEOparse.get_GEO(geo=GEO_ID, silent=True)

# map sample title to metadata
metadata = {}

for key, value in gse.gsms.items():
    sample_title = value.metadata['title'][0]

    metadata_object = {'sample_title': sample_title}
    for item in value.metadata['characteristics_ch1']:
        k, v = item.split(':', 1)
        metadata_object[k.strip()] = v.strip()

    metadata[key] = metadata_object


metadata_df = pd.DataFrame.from_dict(metadata, orient='index')
metadata_df.index.name = 'samples'
# metadata_df = metadata_df.reset_index()

metadata_df = metadata_df[metadata_df['sample_title'].str.contains(r'\(RNA-seq\)', na=False)]

# drop adjacent normal samples (handle normal/tumor samples as needed)
metadata_df = metadata_df[metadata_df['tissue'] == 'Tumor']

metadata_df

,sample_title,patient,sequencing type,tissue,recurrence status,progression type,# _of_cells,progression,recurrence
samples,,,,,,,,,
GSM7173465,NYU1019-T (RNA-seq),NYU1019,NaN,Tumor,NaN,Systemic,NaN,Progression,Recurrence
GSM7173467,NYU1027-T (RNA-seq),NYU1027,NaN,Tumor,NaN,No Progression (>5y),NaN,No Progression (>5y),No Progression (>5y)
GSM7173469,NYU1037-T (RNA-seq),NYU1037,NaN,Tumor,NaN,Short Follow Up (<5y),NaN,N/A,N/A
GSM7173471,NYU1057-T (RNA-seq),NYU1057,NaN,Tumor,NaN,No Progression (>5y),NaN,No Progression (>5y),No Progression (>5y)
GSM7173473,NYU1079-T (RNA-seq),NYU1079,NaN,Tumor,NaN,No Progression (>5y),NaN,No Progression (>5y),No Progression (>5y)
...,...,...,...,...,...,...,...,...,...
GSM7173701,NYU875-T (RNA-seq),NYU875,NaN,Tumor,NaN,No Progression (>5y),NaN,No Progression (>5y),No Progression (>5y)
GSM7173703,NYU947-T (RNA-seq),NYU947,NaN,Tumor,NaN,No Progression (>5y),NaN,No Progression (>5y),No Progression (>5y)
GSM7173705,NYU955-T (RNA-seq),NYU955,NaN,Tumor,NaN,No Progression (>5y),NaN,No Progression (>5y),No Progression (>5y)


In [2]:
SUBSERIES_GEO_ID = 'GSE229705'
# Raw counts
ncbi_data_url = f"https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts&acc={SUBSERIES_GEO_ID}&format=file&file={SUBSERIES_GEO_ID}_raw_counts_GRCh38.p13_NCBI.tsv.gz"

df_raw_counts: pd.DataFrame = pd.read_csv(ncbi_data_url, sep='\t', index_col=0, compression='gzip')
df_raw_counts = df_raw_counts.T
df_raw_counts.index.name = "samples"
df_raw_counts.columns.name = None

url = "https://www.ncbi.nlm.nih.gov/geo/download/?format=file&type=rnaseq_counts&file=Human.GRCh38.p13.annot.tsv.gz"
df_gene_annotations = pd.read_csv(
    url,
    sep="\t",
    compression="gzip"
)
gene_mapping = (
    df_gene_annotations[["GeneID", "Symbol"]]
    .dropna(subset=["GeneID", "Symbol"])
    .set_index("GeneID")["Symbol"]
    .to_dict()
)
# Keep only columns that have a mapping
mappable = [c for c in df_raw_counts.columns if c in gene_mapping]
df_raw_counts = df_raw_counts[mappable].rename(columns=gene_mapping)
# df_raw_counts = df_raw_counts.rename(columns=gene_mapping)

df_combined = metadata_df.join(df_raw_counts, how='inner')
df_combined.reset_index()
# df_combined.to_csv(f'{GEO_ID}_raw_counts_NCBI.csv', index=False)

df_combined

/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_8005/741546085.py:11: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_gene_annotations = pd.read_csv(


,sample_title,patient,sequencing type,tissue,recurrence status,progression type,# _of_cells,progression,recurrence,DDX11L1,...,ND4,TRNH,TRNS2,TRNL2,ND5,ND6,TRNE,CYTB,TRNT,TRNP
samples,,,,,,,,,,,,,,,,,,,,,
GSM7173465,NYU1019-T (RNA-seq),NYU1019,NaN,Tumor,NaN,Systemic,NaN,Progression,Recurrence,9,...,434278,61,43,389,216643,51657,269,197711,72,815
GSM7173467,NYU1027-T (RNA-seq),NYU1027,NaN,Tumor,NaN,No Progression (>5y),NaN,No Progression (>5y),No Progression (>5y),1,...,326777,44,42,287,217919,50870,210,176713,36,632
GSM7173469,NYU1037-T (RNA-seq),NYU1037,NaN,Tumor,NaN,Short Follow Up (<5y),NaN,N/A,N/A,1,...,105021,24,24,135,76342,16504,103,82405,51,313
GSM7173471,NYU1057-T (RNA-seq),NYU1057,NaN,Tumor,NaN,No Progression (>5y),NaN,No Progression (>5y),No Progression (>5y),5,...,235556,108,85,95,126128,28491,250,122385,102,1066
GSM7173473,NYU1079-T (RNA-seq),NYU1079,NaN,Tumor,NaN,No Progression (>5y),NaN,No Progression (>5y),No Progression (>5y),26,...,10902,64,58,57,9424,1015,381,8132,407,3125
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GSM7173701,NYU875-T (RNA-seq),NYU875,NaN,Tumor,NaN,No Progression (>5y),NaN,No Progression (>5y),No Progression (>5y),4,...,401130,225,231,576,307809,59041,503,199930,139,1351
GSM7173703,NYU947-T (RNA-seq),NYU947,NaN,Tumor,NaN,No Progression (>5y),NaN,No Progression (>5y),No Progression (>5y),4,...,657856,39,33,408,335207,72585,132,213398,114,1054
GSM7173705,NYU955-T (RNA-seq),NYU955,NaN,Tumor,NaN,No Progression (>5y),NaN,No Progression (>5y),No Progression (>5y),9,...,372272,82,56,326,241737,48727,367,191895,143,956


In [3]:
# Download supplementary data from: https://pmc.ncbi.nlm.nih.gov/articles/instance/10632519/bin/41467_2023_42327_MOESM4_ESM.xlsx

df_clinical_data = pd.read_excel('41467_2023_42327_MOESM4_ESM.xlsx', sheet_name=1)

df_clinical_data['Study_ID'] = df_clinical_data['Study_ID'].astype(str).str.strip()
df_clinical_data['patient'] = 'NYU' + df_clinical_data['Study_ID']

df_clinical_data.to_csv('41467_2023_42327_MOESM4_ESM.csv', index=False)

df_combined = df_clinical_data.merge(
    df_combined,
    left_on='patient',
    right_on='patient',
    how='inner'
)


data_file_name = f'{GEO_ID}_raw_counts_NCBI.csv'
df_combined.to_csv(data_file_name, index=False)
df_combined



,Study_ID,Race,Male_1,stage,Dead_or_alive,smoking_Pack_Years,Days_to_Death or Last Contact,Age_range,Size_cm,LVI,...,ND4,TRNH,TRNS2,TRNL2,ND5,ND6,TRNE,CYTB,TRNT,TRNP
0,20,White,1,IA,1,50.0,1141,70-79,1.3,0,...,64704,22,7,39,30341,8019,33,28438,25,370
1,32,White,2,IA,0,5.0,4661,60-69,2.0,1,...,66287,12,4,58,38957,9149,45,31996,15,336
2,50,White,1,1,0,7.5,2319,70-79,1.0,0,...,251987,89,73,335,164734,29482,179,167360,101,1129
3,62,White,1,1,0,45.0,4610,60-69,1.4,0,...,217246,33,25,242,170370,30866,160,124072,82,697
4,113,White,2,1,0,35.0,3030,70-79,1.3,0,...,81006,17,19,82,45669,8593,87,43998,20,258
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117,1986,Asian,2,IA,0,0.0,1861,60-69,2.0,0,...,168383,21,16,160,87777,19029,123,82896,20,576
118,1997,Asian,1,IA,0,5.0,1877,50-59,1.8,0,...,92797,25,20,111,61721,15355,119,45856,43,774
119,1999,Asian,1,IA,0,1.0,1903,60-69,2.0,0,...,92537,25,26,149,56401,10663,98,59881,40,323
120,2002,White,2,IA,0,40.5,1801,80-89,2.4,0,...,141336,73,56,292,80546,20710,100,82044,53,541


# Generate description

In [4]:
import json
import os
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent  # adjust based on notebook depth
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from geo.scripts.generate_descriptions import generate_description_from_supplement

description_file_path = 'description.json'

if not os.path.exists(description_file_path):
    with open(description_file_path, 'w') as f:
        json.dump({}, f)

with open(description_file_path, 'r') as f:
    desc_data = json.load(f)

if not desc_data:
    desc_data = generate_description_from_supplement(GEO_ID, '41467_2023_42327_MOESM4_ESM.csv')
    print("Description generated")

desc_data['data_summary'] = {
    'samples': df_clinical_data.shape[0],
    'clinical_features': df_combined.shape[1] - df_raw_counts.shape[1],
    'genes': df_raw_counts.shape[1]
}

print("Description updated")

desc_data['data_file_names'] = [data_file_name]

with open(description_file_path, "w") as f:
    json.dump(desc_data, f, indent=4)


Description updated
